# Recarga limpia del champion

Notebook limpio (sección 8 del PDF): carga `models:/portafolio-inversiones-modelo@champion` desde el Model Registry,
predice el hold-out y reproduce exactamente el MAE registrado en la corrida `final` del champion.

In [ ]:
from pathlib import Path
import sys, os
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np
import mlflow
from mlflow import MlflowClient
from src.datos import cargar
from src.pipeline import preparar, dividir, TARGET
from sklearn.metrics import mean_absolute_error

mlflow.set_tracking_uri(os.environ.get('MLFLOW_TRACKING_URI', 'http://localhost:5001'))
uri = 'models:/portafolio-inversiones-modelo@champion'
modelo = mlflow.pyfunc.load_model(uri)
version = MlflowClient().get_model_version_by_alias('portafolio-inversiones-modelo', 'champion')
obt, meta = cargar()
_, holdout = dividir(preparar(obt))
pred = modelo.predict(holdout[modelo.metadata.get_input_schema().input_names()])
mae = mean_absolute_error(holdout[TARGET], pred)

final = mlflow.search_runs(experiment_names=['portafolio-inversiones'],
                           filter_string="tags.fase = 'final' and tags.rol = 'champion'",
                           order_by=['attributes.start_time DESC']).iloc[0]
registrado = final['metrics.mae_holdout']
print(f'version {version.version} (run {version.run_id}); snapshot {meta["sha256"][:16]}')
print(f'MAE hold-out recalculado: {mae:.10f} | registrado: {registrado:.10f}')
assert np.isclose(mae, registrado, rtol=0, atol=1e-12), 'No se reproduce la métrica registrada'
print('Reproducido exactamente.')